# 10. K-Nearest Neighbours (KNN)## What is this notebook about?KNN is the simplest possible algorithm. To predict a new point:1. Find the K most similar points in the training data.2. Take the majority vote (classification) or average (regression).That's it. There's **no real "training"** - the algorithm just memorizes the training data and uses it at predict time.## What you will learn1. How to use KNN for classification2. Why **feature scaling is critical** for KNN (distances are everything!)3. How to choose **K** with cross-validation4. KNN's strengths (simplicity) and weaknesses (slow at predict time)## DatasetsIris and Wine - both built into sklearn.

In [ ]:
# If you're running locally, uncomment and run this once.# In Google Colab, all of these are pre-installed - you can skip this cell.# !pip install -q numpy pandas scikit-learn matplotlib seaborn scipy xgboost lightgbm

In [ ]:
# Standard imports used in every ML notebookimport numpy as np                 # numerical arraysimport pandas as pd                # tabular data (DataFrames)import matplotlib.pyplot as plt    # plottingimport seaborn as sns              # prettier statistical plots# Set up plot stylesns.set_style("whitegrid")plt.rcParams["figure.figsize"] = (10, 6)# Set a random seed so results are reproduciblenp.random.seed(42)

In [ ]:
from sklearn.datasets import load_iris, load_winefrom sklearn.model_selection import train_test_split, cross_val_scorefrom sklearn.preprocessing import StandardScalerfrom sklearn.pipeline import Pipelinefrom sklearn.neighbors import KNeighborsClassifierfrom sklearn.metrics import accuracy_score

## Step 1. KNN with and without scalingKNN measures distances between points. If one feature has a huge range (income in $) and another has a small range (age in years), the big-range feature dominates. **Always scale before KNN.**

In [ ]:
# Load Irisiris = load_iris(as_frame=True)X, y = iris.data, iris.targetX_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)# WITHOUT scalingknn1 = KNeighborsClassifier(n_neighbors=5).fit(X_train, y_train)acc_raw = knn1.score(X_test, y_test)print(f"Raw features (no scaling):    {acc_raw:.3f}")# WITH scaling, using a Pipelinepipe = Pipeline([    ("scaler", StandardScaler()),    ("knn", KNeighborsClassifier(n_neighbors=5)),])pipe.fit(X_train, y_train)acc_scaled = pipe.score(X_test, y_test)print(f"Scaled features:               {acc_scaled:.3f}")

On Iris, all 4 features are already on similar scales (centimeters), so scaling makes little difference. You'll see a bigger effect on the Wine dataset later.## Step 2. Choose K with cross-validationSmall K = sensitive to noise (overfit). Large K = too smooth (underfit). Find the sweet spot.

In [ ]:
# Try K from 1 to 30ks = range(1, 31)scores = []for k in ks:    pipe = Pipeline([("scaler", StandardScaler()), ("knn", KNeighborsClassifier(n_neighbors=k))])    s = cross_val_score(pipe, X_train, y_train, cv=5).mean()    scores.append(s)plt.figure(figsize=(10, 5))plt.plot(ks, scores, "o-", color="black")plt.xlabel("K (number of neighbours)")plt.ylabel("Cross-validation accuracy")plt.title("Choosing K for KNN (Iris)")plt.show()best_k = ks[np.argmax(scores)]print(f"Best K: {best_k}")

**Common rule of thumb:** start with K = sqrt(n_samples). On Iris that's ≈ 12.## Step 3. KNN on the Wine datasetWine has 13 chemical features with very different scales (some near 1, others near 1000). This is where scaling REALLY matters.

In [ ]:
# Load Winewine = load_wine(as_frame=True)X, y = wine.data, wine.target# Look at the wildly different feature scalesprint("Feature mean values (notice the huge differences):")print(X.mean().round(2))

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)# WITHOUT scaling - typically performs poorlyknn_raw = KNeighborsClassifier(n_neighbors=5).fit(X_train, y_train)print(f"WITHOUT scaling: {knn_raw.score(X_test, y_test):.3f}")# WITH scaling, using weights="distance" so closer neighbours count morepipe = Pipeline([    ("scaler", StandardScaler()),    ("knn", KNeighborsClassifier(n_neighbors=7, weights="distance")),])pipe.fit(X_train, y_train)print(f"WITH scaling:    {pipe.score(X_test, y_test):.3f}")

**Massive jump from scaling.** This is why every KNN tutorial hammers on scaling.## Step 4. Exercises1. **Try different distance metrics:**   ```python   KNeighborsClassifier(n_neighbors=5, metric="manhattan")   ```2. **Try KNN regression** on California Housing (`KNeighborsRegressor`). Is it competitive with linear regression?3. **Plot the decision boundary** of KNN on the first two Iris features. (Use `sklearn.inspection.DecisionBoundaryDisplay`.)4. **Time the prediction** of KNN with 10,000 training points vs 1,000. KNN gets slow on big data.5. **Compare `weights="uniform"` vs `weights="distance"`**. Distance weighting often wins.Next: **11 - End-to-End Titanic** (the capstone)!